# Lab 3 - Model Comparison (F1)

**Framing:** Regression to predict driver race points.

**Business question:** How many constructor points should we expect from each driver this Sunday based only on pre-race information?

**Primary metric:** MAE (lower is better).

**Validation:** Temporal holdout (train: past seasons, test: future season).

In [15]:
import sys
import random
import warnings

import numpy as np
import pandas as pd
import requests

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy : {np.__version__}")
print(f"Seed  : {RANDOM_SEED}")

Python: 3.12.3
NumPy : 2.4.3
Seed  : 414


In [16]:
BASE_URL = "https://api.jolpi.ca/ergast/f1"
SEASONS = [2020, 2021, 2022, 2023, 2024]
TARGET = "points"


def paginate(url_template: str, table_key: str, row_key: str) -> list:
    rows = []
    offset = 0
    limit = 100
    while True:
        url = f"{url_template}?limit={limit}&offset={offset}"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()["MRData"]
        rows.extend(data[table_key][row_key])
        total = int(data["total"])
        offset += limit
        if offset >= total:
            break
    return rows


all_results = []
all_qualifying = []

for year in SEASONS:
    yr_results = paginate(
        f"{BASE_URL}/{year}/results.json",
        table_key="RaceTable",
        row_key="Races",
    )
    yr_qualifying = paginate(
        f"{BASE_URL}/{year}/qualifying.json",
        table_key="RaceTable",
        row_key="Races",
    )
    all_results.extend(yr_results)
    all_qualifying.extend(yr_qualifying)

print(f"Downloaded races with results: {len(all_results)}")
print(f"Downloaded races with qualifying: {len(all_qualifying)}")

rows = []
for race in all_results:
    season = int(race["season"])
    round_no = int(race["round"])
    race_name = race["raceName"]
    circuit = race["Circuit"]["circuitId"]
    race_date = race["date"]

    for res in race["Results"]:
        rows.append(
            {
                "season": season,
                "round": round_no,
                "race_name": race_name,
                "circuit": circuit,
                "race_date": race_date,
                "driver": res["Driver"]["driverId"],
                "constructor": res["Constructor"]["constructorId"],
                "grid": int(res["grid"]),
                "finish_pos": int(res["position"]),
                "points": float(res["points"]),
            }
        )

df_results = pd.DataFrame(rows)

q_rows = []
for race in all_qualifying:
    season = int(race["season"])
    round_no = int(race["round"])
    for q in race.get("QualifyingResults", []):
        def to_sec(x):
            if x is None:
                return np.nan
            if isinstance(x, str) and ":" in x:
                m, s = x.split(":")
                return int(m) * 60 + float(s)
            return np.nan

        q_rows.append(
            {
                "season": season,
                "round": round_no,
                "driver": q["Driver"]["driverId"],
                "constructor_q": q["Constructor"]["constructorId"],
                "quali_pos": int(q["position"]),
                "q3_time_sec": to_sec(q.get("Q3")),
            }
        )

df_qualifying = pd.DataFrame(q_rows)

df = df_results.merge(
    df_qualifying[["season", "round", "driver", "quali_pos", "q3_time_sec"]],
    on=["season", "round", "driver"],
    how="left",
)

df = df.sort_values(["driver", "season", "round"]).reset_index(drop=True)

df["driver_prev_points"] = df.groupby("driver")["points"].shift(1)
df["driver_prev_grid"] = df.groupby("driver")["grid"].shift(1)
df["driver_points_avg_3"] = (
    df.groupby("driver")["points"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
df["driver_top10_rate_3"] = (
    df.groupby("driver")["finish_pos"]
    .transform(lambda s: (s.shift(1) <= 10).astype(float).rolling(3, min_periods=1).mean())
)

df_model = df.dropna(subset=["grid", "points"]).copy()

print(df_model[["season", "round", "driver", "grid", "quali_pos", "points"]].head())
print()
print("Rows by season:")
print(df_model.groupby("season").size())

Downloaded races with results: 111
Downloaded races with qualifying: 115
   season  round  driver  grid  quali_pos  points
0    2020     16  aitken    17       18.0     0.0
1    2020      1   albon     4        5.0     0.0
2    2020      2   albon     6        7.0    12.0
3    2020      3   albon    13       13.0    10.0
4    2020      4   albon    12       12.0     4.0

Rows by season:
season
2020    340
2021    440
2022    440
2023    440
2024    479
dtype: int64


## Temporal split and comparison setup

- Train seasons: 2020, 2021, 2022 and 2023
- Test season: 2024
- Primary metric for every row: MAE
- Two baselines are included:
  1. Global mean points baseline
  2. Domain heuristic baseline from train average points by grid position
- Ridge is tuned over alphas: 0.01, 0.1, 1, 10, 100, 1000 using a temporal validation split inside train

In [17]:
TRAIN_SEASONS = [2020, 2021, 2022, 2023]
TEST_SEASON = 2024

train_df = df_model[df_model["season"].isin(TRAIN_SEASONS)].copy()
test_df = df_model[df_model["season"] == TEST_SEASON].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Temporal split failed: one of the sets is empty.")

feature_cols_num = [
    "grid", "quali_pos", "q3_time_sec",
    "driver_prev_points", "driver_prev_grid", "driver_points_avg_3", "driver_top10_rate_3",
]
feature_cols_cat = ["constructor", "circuit"]
all_feature_cols = feature_cols_num + feature_cols_cat

X_train = train_df[all_feature_cols]
y_train = train_df[TARGET]
X_test = test_df[all_feature_cols]
y_test = test_df[TARGET]


def evaluate_regressor(name: str, model) -> dict:
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": name,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


def evaluate_baseline_constant() -> dict:
    pred_value = y_train.mean()
    pred_train = np.full(shape=len(y_train), fill_value=pred_value)
    pred_test = np.full(shape=len(y_test), fill_value=pred_value)
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": "Baseline 1 - Global mean points",
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


def evaluate_baseline_grid_heuristic() -> dict:
    # Domain heuristic: expected points by starting grid position, learned only from train seasons.
    grid_means = train_df.groupby("grid")[TARGET].mean()
    fallback = y_train.mean()

    pred_train = train_df["grid"].map(grid_means).fillna(fallback)
    pred_test = test_df["grid"].map(grid_means).fillna(fallback)

    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    return {
        "Model": "Baseline 2 - Grid heuristic",
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Gap (Test-Train)": test_mae - train_mae,
    }


preprocess_linear = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), feature_cols_num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_cols_num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), feature_cols_cat),
    ]
)

linreg = Pipeline([
    ("prep", preprocess_linear),
    ("model", LinearRegression()),
])

# Temporal tuning for Ridge: fit on 2020-2022, validate on 2023, then retrain best alpha on 2020-2023.
ridge_train_df = train_df[train_df["season"].isin([2020, 2021, 2022])].copy()
ridge_val_df = train_df[train_df["season"] == 2023].copy()

if ridge_train_df.empty or ridge_val_df.empty:
    raise ValueError("Ridge temporal tuning failed: missing rows for 2020-2022 train or 2023 validation.")

X_ridge_train = ridge_train_df[all_feature_cols]
y_ridge_train = ridge_train_df[TARGET]
X_ridge_val = ridge_val_df[all_feature_cols]
y_ridge_val = ridge_val_df[TARGET]

ridge_alpha_candidates = [0.01, 0.1, 1, 10, 100, 1000]
ridge_val_scores = []

for alpha in ridge_alpha_candidates:
    ridge_candidate = Pipeline([
        ("prep", preprocess_linear),
        ("model", Ridge(alpha=alpha, random_state=RANDOM_SEED)),
    ])
    ridge_candidate.fit(X_ridge_train, y_ridge_train)
    val_pred = ridge_candidate.predict(X_ridge_val)
    val_mae = mean_absolute_error(y_ridge_val, val_pred)
    ridge_val_scores.append((alpha, val_mae))

best_ridge_alpha, best_ridge_val_mae = min(ridge_val_scores, key=lambda x: x[1])

ridge_best = Pipeline([
    ("prep", preprocess_linear),
    ("model", Ridge(alpha=best_ridge_alpha, random_state=RANDOM_SEED)),
])

rf = Pipeline([
    ("prep", preprocess_tree),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=3,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )),
])

rows = []
rows.append(evaluate_baseline_constant())
rows.append(evaluate_baseline_grid_heuristic())
rows.append(evaluate_regressor("Linear Regression", linreg))
rows.append(evaluate_regressor(f"Ridge (best alpha={best_ridge_alpha})", ridge_best))
rows.append(evaluate_regressor("Random Forest (300 trees)", rf))

comparison_df = pd.DataFrame(rows).sort_values("Test MAE", ascending=True).reset_index(drop=True)
comparison_df.insert(0, "Rank", np.arange(1, len(comparison_df) + 1))

for col in ["Train MAE", "Test MAE", "Gap (Test-Train)"]:
    comparison_df[col] = comparison_df[col].round(3)

print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
print(f"Ridge alpha candidates: {ridge_alpha_candidates}")
print(f"Best Ridge alpha (val on 2023): {best_ridge_alpha} | Val MAE: {best_ridge_val_mae:.3f}")
comparison_df

Train rows: 1660 | Test rows: 479
Ridge alpha candidates: [0.01, 0.1, 1, 10, 100, 1000]
Best Ridge alpha (val on 2023): 10 | Val MAE: 3.356


,Rank,Model,Train MAE,Test MAE,Gap (Test-Train)
0,1,Random Forest (300 trees),2.040,2.817,0.777
1,2,Baseline 2 - Grid heuristic,3.485,3.042,-0.443
2,3,Ridge (best alpha=10),3.369,3.216,-0.153
3,4,Linear Regression,3.361,3.269,-0.091
4,5,Baseline 1 - Global mean points,5.885,5.902,0.017


## Why these results

- **Baseline 1 - Global mean points**: Uses one constant prediction for every entry. It ignores grid and constructor context, so it is stable but usually inaccurate.
- **Baseline 2 - Grid heuristic**: Starting grid has a strong relation with points in F1, so this simple domain prior can perform surprisingly well.
- **Linear Regression**: Captures global linear trends but cannot represent sharp nonlinear effects (for example, front-row advantage).
- **Ridge (best tuned alpha)**: Applies linear regularization and the selected alpha is chosen with temporal validation (2020-2022 -> 2023) to improve generalization.
- **Random Forest (300 trees)**: Captures nonlinear interactions among grid, recent form, constructor, and circuit context; watch the train-test gap for overfitting risk.

## Interpretation

- The best model should be selected by **lowest Test MAE**, not by train score.
- The **Train-Test gap** diagnoses overfitting risk.
- If a baseline wins, that is still a valid and useful result: it means current features/models are not adding enough predictive value yet.